# 💻 💰 Calculating Accrued Interest
<br>
<br>


<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Three-Chapter-Two/blob/master/colab/Colab_Chapter_Two_Accrued_Interest.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Three-Chapter-Two/master?urlpath=lab/tree/notebooks/Chapter_Two_Accrued_Interest.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Three-Chapter-Two/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

</div>

<details>
<summary><b>🤔 Notebook Setup: Why the "Try/Except" Imports?</b></summary>

**The Goal:**
To ensure this notebook runs perfectly whether you are using **Google Colab**, a local **Jupyter instance**, or a remote server without you having to manually install software.

**Key Concepts in this Section:**
*   **Standard Libraries:** Modules like `os`, `sys`, and `datetime` come "in the box" with Python. We use them for system tasks and date math.
*   **External Libraries:** NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.
*   **The `try/except` Logic:** This is a safety net.
    1. We **try** to import the library.
    2. If it fails (because it's not installed), the **except** block triggers a `%pip install` to download it automatically.
*   **Aliasing (`as np`):** We rename `numpy` to `np` to save keystrokes. In professional finance code, `np` and `pd` are the universal shorthand.

</details>

## 🛠️ Preparing the Notebook

<details>
<summary><b>👉 Click to Expand: 📦 Importing Libraries, Modules, and Functions</b></summary>

As a best practice, we always begin by importing our necessary dependencies in the very first code cell. Notice how lightweight our imports are here: just the built-in `datetime` module and the `pandas` library. ✨

This simplicity is a direct result of using the `financial_quant` package, which handles the heavy lifting behind the scenes. 🏗️ By keeping complicated setup details out of sight, we ensure the spotlight remains focused exactly where it belongs—on the core analysis. 🎯

```python
from datetime import date, datetime

try:
    import pandas as pd
except ImportError:
    %pip install pandas
    import pandas as pd
```
**👀 Keep an eye out**: As we progress, pay attention to how the `financial_quant` package is imported as `fq`, and how every reference to its functions begins with fq.. 💡 This follows the exact same standard practice we demonstrated in Chapter One with NumPy (np) and Pandas (pd).

</details>

In [1]:

from datetime import date, datetime

try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd



## 📦 Getting Functions from financial_quant package

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">🔌 Dynamic Verification: How Remote Version Checking Works</b></summary>
<br>
<p><b>The Logic:</b><br>
Instead of blindly reinstalling packages or guessing if our local tools are up to date, we are dynamically reaching out to GitHub to inspect the live code. By scanning the remote file on the fly, the notebook can verify exactly which version of the quantitative toolkit is currently published before we use it.</p>

<p><b>The Workflow:</b></p>

<ol>
<li><b>Request:</b> The code acts like an automated web browser. It uses Python's `urllib` to visit the GitHub URL and request the raw text of the Python script.</li>
<li><b>Decode:</b> The internet sends data back as raw bytes (1s and 0s). We use `.decode('utf-8')` to translate that raw data into a readable Python string.</li>
<li><b>Search & Extract:</b> We use a Regular Expression (`regex`) to scan the document, pinpointing the exact line where the version is declared and extracting the number.</li>
<li><b>Verify:</b> The notebook compares this remote version against your local installation to ensure you have the latest financial models. </li>
<li><b>Import & Route:</b> Once verified, we `import financial_quant as fq`. Behind the scenes, an `__init__.py` file routes the complex tools from deeply nested folders up to the surface.</li>
<li><b>Execute:</b> You can now confidently type `fq.one_y_axis()` or `fq.calc_ytm()`, knowing you are running the most up-to-date code.</li>
</ol>

<p><b>Why do this?</b><br>
It makes your notebooks highly intelligent. Rather than relying on static installations, the notebook becomes self-aware—it can check if its math models are perfectly synced with the master branch and alert you if a critical update is missing. It is a lightweight way to guarantee reproducibility without cluttering the environment.</p>
</details>

In [14]:
import urllib.request

updater_url = "https://raw.githubusercontent.com/PatrickJHess/quant_repo/master/fq_updater.py"
exec(urllib.request.urlopen(updater_url).read().decode('utf-8'))

fq = import_financial_quant()

✅ 'financial_quant' is up to date (Version 1.0.1).


## 🗓 Scheduled payment dates

To calculate a bond's accrued interest, the next scheduled payment date must be known. Because a bond's maturity date is a scheduled payment date, all other scheduled payment dates can be determined by working backward from the maturity date, using the time interval between payments. The `scheduled_pay_dates` function performs this calculation.

The function uses a `while` loop that starts on or before the maturity date and moves backward toward the settlement date. The starting date must have the same month and day of the month as the maturity date (e.g. a February 28<sup>th</sup> 2030 maturity date would result in a 2028 start date of February 29<sup>th</sup>). In each loop iteration, a scheduled payment date is added to the `dates` list.

The time between payments is specified by `num_months` (e.g., six for semi-annual payments). The initial value of `pay_date` is the starting date, and it is reduced by `num_months` in each subsequent iteration. The variable `is_month_end` is set using `calendar.monthrange` (as demonstrated in the *Manipulating Dates* notebook) to ensure that if the start date is a month-end, all scheduled payments are also month-end.

The dates are initially generated in reverse chronological order because the calculation starts from a future date. To correct the sequence, the list is reversed using the `[::-1]` slice notation ([more info on list slicing](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/A_First_Look_At_Lists.html#combining-and-slicing-lists-with-the-index)).

The only required argument for the function is the starting date (`last_date`), which must be a `datetime` or `date` object. By default, the settlement date is set to the current date, and the payment frequency is '2' for semi-annual payments.


<details>

<summary style="cursor: pointer; font-size: 1.1em; color: #1a237e;"><b>🔍 Click to see the <code>scheduled_pay_dates</code> function</b></summary>

```py
def scheduled_pay_dates(last_date,settlement=None,freq=2):
  '''
    Generates a chronological list of coupon payment dates from settlement to last_date.
    The function calculates dates backward from the last_date date based on the
    specified frequency. It handles standard bond market "end-of-month" logic:
    if the last_date date is the last day of a month, all preceding coupon payments
    are snapped to the last day of their respective months.
    Args:
        last_date (datetime.date): The final last_date date of the bond.
            Accepts a date object.
        settlement (datetime.date, optional): The settlement date (start of analysis).
            Coupons falling before this date are excluded. Defaults to date.today().
        freq (int, optional): The number of coupon payments per year.
            Accepted values:
            * 1: Annual, 2: Semi-Annual (Default), 4: Quarterly, 12: Monthly

    Returns:
        list[datetime.date]: A list of coupon dates sorted chronologically
        (earliest to latest), ending with the last_date date..
  '''
  from datetime import datetime,date
  import calendar

  from IPython.display import Markdown as md, display

  from dateutil.relativedelta import relativedelta

  # check for datetime object and convert to datetime to date
  def validate_date(datetime_object):
      # check for datetime or date
      if not isinstance(datetime_object, (datetime, date)):
          raise TypeError("Input must be a datetime or date object.")
      # convert datetime to date
      if isinstance(datetime_object, datetime):
        datetime_object = datetime_object.date()
      return datetime_object
      
  #Validate the data- last_date, coupon, settlement, freq
  #last_date
  last_date=validate_date(last_date)

  #settlement
  if settlement is None:
      settlement = date.today()
  else:
      settlement=validate_date(settlement)
  #freq
  if int(freq) not in [1,2,4,12]:
      display(md(f"### ⚠️  your assigned freq {freq} it must be (1, 2, 4, or 12)\
     \n     semi-annual assumed (2)."))
      freq=int(2)

  # check last_date greater than settlement
  if last_date<=settlement:
    raise ValueError("last_date must be greater than the settlement date")

  # Calculate the number of months between each coupon payment.
  num_months=int(12/freq)

  #Need to check for month_end
  is_month_end = last_date.day == calendar.monthrange(last_date.year, last_date.month)[1]

  dates = []
  pay_date = last_date

  #Loop backward from the last_date date

  while pay_date > settlement:
    dates.append(pay_date)

    # Decrement by the frequency
    pay_date -= relativedelta(months=num_months)

    # Handle Month End Logic
    if is_month_end:
      last_day = calendar.monthrange(pay_date.year, pay_date.month)[1]
      pay_date = date(pay_date.year, pay_date.month, last_day)

  # Return chronologically (sliced backward)
  return dates[::-1]
```
</details>






In [3]:
# @title ✂️ Click show code for `scheduled_pay_dates` snippets or execute
from datetime import datetime,date
# last_date and settlement dates
last_date=date(2027,3,15)
settlement=date(2025, 3, 17)
# get scheduled payment dates
scheduled_payments=fq.scheduled_pay_dates(last_date,settlement=settlement,freq=2)
display(scheduled_payments)

array([datetime.date(2025, 9, 15), datetime.date(2026, 3, 15),
       datetime.date(2026, 9, 15), datetime.date(2027, 3, 15)],
      dtype=object)

## % Accrued interest calculation requires two or three dates: settlement, last payment, and next payment
To calculate accrued interest, it is essential to determine the next or last payment date. For U.S. Treasury bonds or notes, both dates are required. These payment dates are established by the bond's maturity date.

The scheduled_pay_dates function is used to calculate payment dates.
If the maturity date is assigned to the variable last_date, the function returns all future payment dates.
Assigning a value earlier than the maturity date returns payment dates between the settlement date and the last_date.
To find the next payment date relative to the settlement date, you can set the start date to the settlement year plus one, using the month and day of the maturity date, while accounting for the last day of the month. The first date returned by scheduled_pay_dates is the next payment date, which can then be used to find the last (or previous) payment date.

A code snippet demonstrating this process is available for testing. Note that our accrued interest calculations assume settlement dates do not fall on a weekend or a holiday.



<details>
<summary style="cursor: pointer; font-size: 1.1em; color: #1a237e;"><b>✂️ Copy the snippet to find the next and the previous payment dates</b></summary>

```python
from datetime import date
# maturity and settlement dates
maturity=date(2027,3,15)
settlement=date(2025, 3, 17)

# semi annual
freq=2

# determine last day of the month adjustment
mat_is_last=maturity.day==calendar.monthrange(maturity.year,maturity.month)[1]
if mat_is_last:
   lastDay=calendar.monthrange(settlement.year+1,maturity.month)[1]
   strategic_date=date(settlement.year+1,maturity.month,lastDay)
else:
   strategic_date=date(settlement.year+1,maturity.month,maturity.day)

# the strategic date: minimum of actual and next year's maturity date
strategic_date=min(maturity,strategic_date)

# next date is the first value returned
next_coupon=fq.scheduled_pay_dates(strategic_date, settlement=settlement, freq=freq)[0]
num_months = int(12 // freq)

# previous coupon date calculated from next date
prev_coupon = next_coupon - relativedelta(months=num_months)

# if necessary adjust for last day of the momth
if mat_is_last:
  prev_last_day=calendar.monthrange(prev_coupon.year,prev_coupon.month)[1]
  prev_coupon=date(prev_coupon.year,prev_coupon.month,prev_last_day)

display(f'Next coupon {next_coupon}..Last coupon {prev_coupon}')
```
</details>

## ❶ ❷ ❸ Calculating accrued interest: counting the days

Calculating the number of days between two dates is relatively simple for Treasury securities using the Actual/Actual convention or for Floating Rate Notes with the Actual/360 convention. Day count calculations become more complex when using Actual/365 (also known as Actual/Actual ISDA) or the 30/360 convention.

To address this complexity, two dedicated functions have been developed. The `convert_isda` function specifically adjusts the denominator for leap years; the `30_360` function calculates the day count according to the 30/360 convention. Both of these functions return the ratio of the coupon that applies to accrued interest.

Both function have been imported.


<details>

<summary style="cursor: pointer; font-size: 1.1em; color: #1a237e;"><b> 🔍 Click to see the <code>convert_isda</code>function</b></summary>

```py
def convert_isda(settlement, prev_coupon):
    from datetime import date
    import calendar
    
    # Easy case: no leap years / same year
    if prev_coupon.year == settlement.year:
        days_in_year = 366 if calendar.isleap(settlement.year) else 365
        actual_days = (settlement - prev_coupon).days
        return actual_days / days_in_year

    # ISDA Split: Pivot on Jan 1st of the second year
    pivot_date = date(settlement.year, 1, 1)
    
    # All days from the coupon up to midnight Jan 1st
    days_prev = (pivot_date - prev_coupon).days
    days_prev_year = 366 if calendar.isleap(prev_coupon.year) else 365
    prev_ratio = days_prev / days_prev_year
    
    # All days from Jan 1st to the accrual end
    days_settlement = (settlement- pivot_date).days
    days_in_settlement_year = 366 if calendar.isleap(settlement.year) else 365
    settlement_ratio = days_settlement / days_in_settlement_year

    return prev_ratio + settlement_ratio
 ```

</details>


<details>
<summary style="cursor: pointer; font-size: 1.1em; color: #1a237e;"><b>🔍 Click to see the <code>_30_360_ </code> function</b></summary>

```py
def _30_360_(settlement,prev_coupon):

    from datetime import date, timedelta
    import calendar
    
    # initalize
    number_days=0

    # accounting for years
    number_days+=(settlement.year-prev_coupon.year)*360

    # accounting for months
    number_days+=(settlement.month-prev_coupon.month)*30
   
    # accounting for days
    # February is the exception
    # Function to check if a date is the last day of its month (Feb 28 or 29)
    def feb_end(date_value):
        # If month is 2, check if it's the last day using calendar.monthrange
        return date_value.month == 2 and date_value.day == calendar.monthrange(date_value.year,
                                                                               date_value.month)[1]
    # STEP 1: Normalize Previous Coupon Date
    # Rule: If it's the 31st OR the end of Feb, it's 30.
    if prev_coupon.day == 31 or feb_end(prev_coupon):
        prev_coupon_days = 30
    else:
        prev_coupon_days = prev_coupon.day
    
    # STEP 2: Conditional Settlement Date
    # Rule: If it's the 31st OR the end of Feb AND the start was 30, it's 30.
    if (settlement.day == 31 or feb_end(settlement)) and prev_coupon_days == 30:
        settlement_days = 30
    else:
        settlement_days = settlement.day
    
    number_days+=settlement_days-prev_coupon_days

    return number_days/360
```
</details>




## ⚙️ The function `accrued_interest`
The function calculates accrued interest using `scheduled_pay_dates`, `convert_isda`, and `_30_360`.

**Required Arguments:**

* `maturity`  
* `coupon`

**Optional Arguments (with Default Values):**

* `day_type`: Defaults to `'Actual/Actual'`. Other acceptable values are `'30/360'`
`'Actual/365'` (equivalent to `'Actual/Actual ISDA'`), and `'Actual/360'`. An invalid value will cause the function to fall back to `'Actual/Actual'`.  
* `settlement`: Defaults to `None`, which results in the current day being used.
<br>

`maturity` and `settlement` must be date or datetime.

<br>

<details>

<summary style="cursor: pointer; font-size: 1.1em; color: #1a237e;"><b>🔍 Click to see the <code>accrued_interest</code> function</b></summary>


```py
def accrued_interest(maturity, coupon, day_type='Actual/Actual', settlement=None, freq=2):
    """
      Returns the accrued interest for a bond.  Returns the accrued interest for a bond.

    Args:
        maturity (datetime): The maturity date of the bond.
        coupon (float): The annual coupon rate (e.g., 0.05 for 5%).
        day_types:
            Actual/Actual, Actual/365, Actual/360. and 30/360.
        settlement (datetime, optional): The settlement date. Defaults to today.
        freq (int, optional):
        Coupon frequency per year: 1 (annual). 2 (semi-annual)
                                    4 (quarterly), 12 (monthly)
                                    Defaults to 2.
    """
    from datetime import datetime,date
    import calendar
    from dateutil.relativedelta import relativedelta
    #Validate Data
    def validate_date(datetime_object):
      # check for datetime or date
      if not isinstance(datetime_object, (datetime, date)):
          raise TypeError("Input must be a datetime or date object.")
      # convert datetime to date
      if isinstance(datetime_object, datetime):
        datetime_object = datetime_object.date()
      return datetime_object
    maturity = validate_date(maturity)

    if settlement is None:
        settlement = date.today()
    else:
        settlement = validate_date(settlement)

    if freq not in [1, 2, 4, 12]:
        print(f"⚠️ Warning: Freq {freq} invalid. Assumed Semi-Annual (2).")
        freq = 2

    try:
        coupon = float(coupon)
        if coupon < 0: raise ValueError
    except:
        raise ValueError("Coupon must be a positive number.")

    # Define strategic_date
    # Get all the bond's potential payment dates in the next year
    mat_is_last=maturity.day==calendar.monthrange(maturity.year,maturity.month)[1]
    if mat_is_last:
      lastDay=calendar.monthrange(settlement.year+1,maturity.month)[1]
      strategic_date=date(settlement.year+1,maturity.month,lastDay)
    else:
     strategic_date=date(settlement.year+1,maturity.month,maturity.day)

    #The strategic date: minimum of actual and next year's maturity date
    strategic_date=min(maturity,strategic_date)
    pay_dates = scheduled_pay_dates(strategic_date, settlement=settlement, freq=freq)

    # Should sorted but check
    pay_dates.sort()

    # The first date after the settlement date is the next coupon date
    next_coupon = None
    for d in pay_dates:
        if d >= settlement:
            next_coupon = d
            break

    # Bond has matured or annual coupon is zero
    if next_coupon is None or coupon==0:
        return 0.0

    #Calculate Previous Coupon Date
    num_months = int(12 // freq)
    prev_coupon = next_coupon - relativedelta(months=num_months)

    # Check for Month End adjustment on the calculated previous date
    is_next_month_end = next_coupon.day == calendar.monthrange(maturity.year, maturity.month)[1]

    if is_next_month_end:
        last_day_of_prev_month = calendar.monthrange(prev_coupon.year, prev_coupon.month)[1]
        prev_coupon = date(prev_coupon.year, prev_coupon.month, last_day_of_prev_month)

    # The day a coupon is paid is also the first day of the new cycle.

    accrued_value = 0.0

    if day_type == 'Actual/Actual':
        days_since_last = (settlement - prev_coupon).days
        days_between = (next_coupon - prev_coupon).days
        #  (DaysHeld / DaysInPeriod)
        accural_ratio= days_since_last/days_between
        # Actual/Actual uses the coupon paid on the date
        accrued_value = (coupon / freq) * accural_ratio


    elif day_type == '30/360':
        accural_ratio =_30_360_(settlement,prev_coupon)
        # Formula: Coupon * accrural ratio
        accrued_value = coupon * accural_ratio

    elif day_type == 'Actual/360':
        days_since_last = (settlement - prev_coupon).days
        accural_ratio= days_since_last/360
      # Formula: Coupon * accrural ratio        
        accrued_value = coupon * accural_ratio

    elif day_type == 'Actual/365':
        accural_ratio = convert_isda(settlement,prev_coupon)
      # Formula: Coupon * accrural ratio        
        accrued_value = coupon * accural_ratio

    else:
        # Fallback
        print(f"⚠️ Warning: Unknown day_type {day_type}. Using Actual/Actual.")
        days_since_last = (settlement - prev_coupon).days
        days_between = (next_coupon - prev_coupon).days
        #  (DaysHeld / DaysInPeriod)
        accural_ratio= days_since_last/days_between
        # Actual/Actual uses the coupon paid on the date
        accrued_value = (coupon / freq) * (days_since_last / days_between)

    return accrued_value
 ```
 </details>

 **✍️ Application: `accrued_interest` for a Treasury bond**
 
 Consider a Treasury bond with a 5.0% coupon that matures on September 30, 2030. Assume a settlement date of May 8, 2026 and calculate the accrued interest.

<details>
<summary>✅ <b>Example Of Solution</b></summary>

### Example of Code

```python
 maturity = datetime(2026,9,30)
 coupon = 5.0
 bond_accrued_interest = fq.accrued_interest(maturity, coupon, day_type='Actual/Actual', settlement=datetime(2026, 5, 8), freq=2)
 display(bond_accrued_interest)
```

</details>

### 💡 ✍️ Check Your Understanding Of Accrued Interest

**Questions:**
Assume that the settlement date is March 17<sup>th</sup> 2025.

1. What is the accrued interest for Dell International LLC 5.5% 01-APR-2035?
2. What would be the accrued interest for a U.S. Treasury security with identical terms?
3. If the settlement date was October 10<sup>th</sup> 2024, would accrued interest be higher or lower?

---

<details>
<summary style="cursor: pointer; color: #2196f3; font-weight: bold;">👉 Click here to reveal the answers</summary>
<div style="margin-top: 10px; padding: 10px; border-left: 3px solid #2196f3; background-color: #f9f9f9;">

**Answers:**
* For the Dell bond the day_type is <code>30/360</code> and accrued interest is **\$2.5208**.
* For Treasury securities the day_type is <code>Actual/Actual</code> and accrued interest is **\$2.5234**.
* Because the bond pays semi-annual coupons, payments are made on April 1<sup>st</sup> and October 1<sup>st</sup>. On March 17<sup>th</sup> there is less than two weeks before the next coupon so most of the coupon is assigned to the seller. On October 10<sup>th</sup>, there are only ten days since the coupon payment and the seller is due a much smaller amount (**\$0.122** for Dell, **\$0.136** for Treasury securities).

</div>
</details>


<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Three-Chapter-Two/blob/master/colab/Colab_Chapter_Two_Accrued_Interest.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Three-Chapter-Two/master?urlpath=lab/tree/notebooks/Chapter_Two_Accrued_Interest.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Three-Chapter-Two/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

</div>